In [ ]:
!pip install torch torchvision torchaudio --quiet
!pip install torch-geometric --quiet
!pip install sentence-transformers --quiet
!pip install pandas scikit-learn --quiet


In [ ]:
import importlib
import subprocess
import sys

import torch


def _ensure_pyg_neighbor_backend():
    """NeighborLoader's sampler needs pyg_lib or torch_sparse (not always pulled in with torch-geometric)."""
    for name in ("pyg_lib", "torch_sparse"):
        try:
            importlib.import_module(name)
            return
        except ImportError:
            continue
    ver = torch.__version__.split("+")[0]
    if torch.cuda.is_available() and torch.version.cuda:
        major, minor = torch.version.cuda.split(".")[:2]
        cx = f"cu{major}{minor}"
    else:
        cx = "cpu"
    index = f"https://data.pyg.org/whl/torch-{ver}+{cx}.html"
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "pyg_lib", "torch_sparse", "-f", index, "-q"]
    )


_ensure_pyg_neighbor_backend()


In [ ]:
# Twitter — training/train_twitter.py: SNAP ego-Twitter (or Kaggle csv), structural features, quartile labels
# Copy Kaggle add-on SNAP files into PyG’s ego-twitter/raw/ so PyG does not download twitter.tar.gz.
import glob
import logging
import os
import shutil

import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
from torch_geometric.datasets import SNAPDataset
from torch_geometric.utils import to_undirected

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

SNAP_ROOT = os.environ.get("SNAP_DATA_ROOT", "/kaggle/working/snap_twitter")
TWITTER_SNAP_SOURCE = os.environ.get(
    "TWITTER_SNAP_SOURCE",
    "/kaggle/input/datasets/akshatsharma1011/twitter/twitter",
)

FEATURE_DIM = 128
LABEL_NAMES = {0: "lurker", 1: "regular", 2: "active", 3: "influencer"}


def _ego_twitter_raw_dir() -> str:
    return os.path.join(SNAP_ROOT, "ego-twitter", "raw")


def prepare_snap_twitter_raw() -> bool:
    """Copy *.circles, *.edges, *.egofeat, *.feat, *.featnames into raw/; drop stale processed data.pt."""
    if not os.path.isdir(TWITTER_SNAP_SOURCE):
        logger.info("Optional SNAP source folder not found: %s", TWITTER_SNAP_SOURCE)
        return False
    raw_dir = _ego_twitter_raw_dir()
    os.makedirs(raw_dir, exist_ok=True)
    n = 0
    for path in glob.glob(os.path.join(TWITTER_SNAP_SOURCE, "*")):
        if os.path.isfile(path):
            shutil.copy2(path, os.path.join(raw_dir, os.path.basename(path)))
            n += 1
    processed_pt = os.path.join(SNAP_ROOT, "ego-twitter", "processed", "data.pt")
    if os.path.isfile(processed_pt):
        os.remove(processed_pt)
    logger.info("Copied %d files from %s -> %s", n, TWITTER_SNAP_SOURCE, raw_dir)
    return n > 0


def _raw_dir_has_files() -> bool:
    d = _ego_twitter_raw_dir()
    if not os.path.isdir(d):
        return False
    return any(os.path.isfile(os.path.join(d, f)) for f in os.listdir(d))


def assign_community_labels(edge_index: torch.Tensor, num_nodes: int) -> torch.Tensor:
    degrees = torch.zeros(num_nodes)
    degrees.scatter_add_(0, edge_index[0], torch.ones(edge_index.size(1)))
    degrees.scatter_add_(0, edge_index[1], torch.ones(edge_index.size(1)))
    q33 = degrees.quantile(0.33).item()
    q66 = degrees.quantile(0.66).item()
    q90 = degrees.quantile(0.90).item()
    labels = torch.zeros(num_nodes, dtype=torch.long)
    labels[degrees > q33] = 1
    labels[degrees > q66] = 2
    labels[degrees > q90] = 3
    return labels


def build_structural_features(edge_index: torch.Tensor, num_nodes: int, feature_dim: int) -> torch.Tensor:
    deg = torch.zeros(num_nodes)
    deg.scatter_add_(0, edge_index[0], torch.ones(edge_index.size(1)))
    deg.scatter_add_(0, edge_index[1], torch.ones(edge_index.size(1)))
    deg_norm = (deg - deg.mean()) / (deg.std() + 1e-8)
    remaining = feature_dim - 1
    rnd = torch.randn(num_nodes, remaining) * 0.1
    return torch.cat([deg_norm.unsqueeze(1), rnd], dim=1)


def load_twitter_ego():
    copied = prepare_snap_twitter_raw()

    kaggle_path = "/kaggle/input/twitter-social-circles/twitter_edges.csv"
    if (not _raw_dir_has_files()) and os.path.exists(kaggle_path):
        df = pd.read_csv(kaggle_path)
        src = torch.tensor(df.iloc[:, 0].values, dtype=torch.long)
        dst = torch.tensor(df.iloc[:, 1].values, dtype=torch.long)
        edge_index = to_undirected(torch.stack([src, dst]))
        num_nodes = int(edge_index.max().item()) + 1
        x = build_structural_features(edge_index, num_nodes, FEATURE_DIM)
        y = assign_community_labels(edge_index, num_nodes)
        return Data(x=x, edge_index=edge_index, y=y, num_nodes=num_nodes)

    logger.info("Loading SNAPDataset from %s (force_reload=%s)", _ego_twitter_raw_dir(), copied)
    dataset = SNAPDataset(root=SNAP_ROOT, name="ego-Twitter", force_reload=copied)
    all_edges_src, all_edges_dst = [], []
    offset = 0
    for d in dataset:
        if d.edge_index.size(1) == 0:
            continue
        all_edges_src.append(d.edge_index[0] + offset)
        all_edges_dst.append(d.edge_index[1] + offset)
        offset += d.num_nodes
    edge_src = torch.cat(all_edges_src)
    edge_dst = torch.cat(all_edges_dst)
    edge_index = to_undirected(torch.stack([edge_src, edge_dst], dim=0))
    num_nodes = offset
    x = build_structural_features(edge_index, num_nodes, FEATURE_DIM)
    y = assign_community_labels(edge_index, num_nodes)
    return Data(x=x, edge_index=edge_index, y=y, num_nodes=num_nodes)


data_pyg = load_twitter_ego()
nodes = list(range(data_pyg.num_nodes))
Y = data_pyg.y.numpy().astype(int)
labels_dict = {i: LABEL_NAMES[int(Y[i])] for i in nodes}
edges = pd.DataFrame({
    "id_1": data_pyg.edge_index[0].numpy(),
    "id_2": data_pyg.edge_index[1].numpy(),
})
X = data_pyg.x.numpy()
print("Nodes:", data_pyg.num_nodes, "X:", X.shape, "edges listed:", len(edges))


In [ ]:
import os
print("SNAP root:", os.environ.get("SNAP_DATA_ROOT", "/kaggle/working/snap_twitter"))


In [ ]:
import pandas as pd
print(edges.head())
print("Total edge rows:", len(edges))


In [ ]:
import numpy as np
print("Classes:", len(np.unique(Y)))


In [ ]:
print("Feature shape:", X.shape)


In [ ]:
documents = []
for node in nodes:
    role = labels_dict[node]
    doc = (
        f"Node ID: {node}\n"
        f"This is a Twitter user in an ego network (activity tier: {role}).\n"
        f"It is connected to other users by follow relationships.\n"
    )
    documents.append(doc)
print("Sample document:", documents[0])


In [ ]:
import torch
from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer("all-MiniLM-L6-v2")
text_embeddings = encoder.encode(documents, show_progress_bar=True, batch_size=64)
text_embeddings = np.array(text_embeddings)
print("Text embedding shape:", text_embeddings.shape)


In [ ]:
import torch
node_to_idx = {n: n for n in nodes}
edge_list = []
for _, row in edges.iterrows():
    src_id = int(row["id_1"])
    dst_id = int(row["id_2"])
    if src_id in node_to_idx and dst_id in node_to_idx:
        src, dst = node_to_idx[src_id], node_to_idx[dst_id]
        edge_list.append([src, dst])
        edge_list.append([dst, src])
edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
print("Edge index shape:", edge_index.shape)


In [ ]:
from torch_geometric.data import Data
import numpy as np

X_combined = np.concatenate([X, text_embeddings], axis=1)
x = torch.tensor(X_combined, dtype=torch.float)
y = torch.tensor(Y, dtype=torch.long)
data = Data(x=x, edge_index=edge_index, y=y)
print(data)


In [ ]:
from sklearn.model_selection import train_test_split
import torch

indices = list(range(len(Y)))
train_idx, test_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=Y
)
train_mask = torch.zeros(len(Y), dtype=torch.bool)
test_mask = torch.zeros(len(Y), dtype=torch.bool)
train_mask[train_idx] = True
test_mask[test_idx] = True
data.train_mask = train_mask
data.test_mask = test_mask


In [ ]:
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
import torch.nn as nn

class GNN_RAG_Model(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_classes):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.bn1 = nn.BatchNorm1d(hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.bn2 = nn.BatchNorm1d(hidden_channels)
        self.conv3 = SAGEConv(hidden_channels, hidden_channels)
        self.bn3 = nn.BatchNorm1d(hidden_channels)
        self.fusion = nn.Linear(hidden_channels, hidden_channels)
        self.classifier = nn.Linear(hidden_channels, num_classes)

    def forward(self, x, edge_index):
        h = F.relu(self.bn1(self.conv1(x, edge_index)))
        h = F.dropout(h, p=0.3, training=self.training)
        h = F.relu(self.bn2(self.conv2(h, edge_index)))
        h = F.dropout(h, p=0.3, training=self.training)
        h = F.relu(self.bn3(self.conv3(h, edge_index)))
        h = F.relu(self.fusion(h))
        return self.classifier(h)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Avoid Y.tolist()+set: on 100k+ nodes that is very slow. Labels are 0..C-1.
num_classes = int(Y.max()) + 1
model = GNN_RAG_Model(
    in_channels=int(data.num_features), hidden_channels=128, num_classes=num_classes
).to(device)
# Keep `data` on CPU; use NeighborLoader to stream subgraphs to GPU (full-batch OOM on large Twitter graphs).
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=50)


In [ ]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.loader import NeighborLoader
from sklearn.metrics import f1_score, roc_auc_score

OUTPUT_DIR = "weights"
os.makedirs(OUTPUT_DIR, exist_ok=True)
best_acc = 0.0

# 3x SAGEConv + neighbor sampling (not full graph — avoids OOM on ~100k+ nodes / millions of edges)
GNN_NUM_NEIGHBORS = (20, 15, 10)
BATCH_SIZE = int(os.environ.get("GNN_BATCH_SIZE", "1024"))
if device.type == "cpu":
    GNN_NUM_NEIGHBORS = (5, 5, 5)
    BATCH_SIZE = 256

train_idx = data.train_mask.nonzero(as_tuple=True)[0]
test_idx = data.test_mask.nonzero(as_tuple=True)[0]
train_loader = NeighborLoader(
    data,
    num_neighbors=GNN_NUM_NEIGHBORS,
    batch_size=BATCH_SIZE,
    input_nodes=train_idx,
    shuffle=True,
)
test_loader = NeighborLoader(
    data,
    num_neighbors=GNN_NUM_NEIGHBORS,
    batch_size=max(BATCH_SIZE, 1024) * 2,
    input_nodes=test_idx,
    shuffle=False,
)


@torch.inference_mode()
def evaluate():
    model.eval()
    y_all, p_all, proba_list = [], [], []
    for batch in test_loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index)[: batch.batch_size]
        y_all.append(batch.y[: batch.batch_size].cpu().numpy())
        p_all.append(out.argmax(1).cpu().numpy())
        proba_list.append(torch.softmax(out, dim=1).cpu().numpy())
    y_te = np.concatenate(y_all)
    p_te = np.concatenate(p_all)
    proba = np.concatenate(proba_list)
    acc = float((p_te == y_te).mean())
    f1v = f1_score(y_te, p_te, average="macro", zero_division=0)
    c = proba.shape[1]
    try:
        if c == 2:
            aucv = roc_auc_score(y_te, proba[:, 1])
        else:
            aucv = roc_auc_score(
                y_te, proba, multi_class="ovr", average="macro", labels=np.arange(c)
            )
    except ValueError:
        aucv = float("nan")
    return acc, f1v, aucv


def train_epoch():
    model.train()
    tot, n = 0.0, 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)[: batch.batch_size]
        loss = F.cross_entropy(out, batch.y[: batch.batch_size])
        loss.backward()
        optimizer.step()
        tot += loss.item() * batch.batch_size
        n += int(batch.batch_size)
    return tot / max(n, 1)


NUM_EPOCHS = int(os.environ.get("GNN_EPOCHS", "101"))
for epoch in range(NUM_EPOCHS):
    loss = train_epoch()
    scheduler.step(loss)
    if epoch % 10 == 0:
        acc, f1v, aucv = evaluate()
        print(
            f"Epoch {epoch:4d} | Loss: {loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f} "
            f"| Test Acc: {acc:.4f} | Test F1: {f1v:.4f} | Test AUC: {aucv:.4f}"
        )
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), f"{OUTPUT_DIR}/best.pth")


In [ ]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score, roc_auc_score
from torch_geometric.loader import NeighborLoader

GNN_NUM_NEIGHBORS = (20, 15, 10)  # match training cell
BATCH = int(os.environ.get("GNN_BATCH_SIZE", "1024")) * 2
test_idx = data.test_mask.nonzero(as_tuple=True)[0]
test_loader = NeighborLoader(
    data, num_neighbors=GNN_NUM_NEIGHBORS, batch_size=max(BATCH, 2048), input_nodes=test_idx, shuffle=False
)


@torch.inference_mode()
def test():
    model.eval()
    y_all, p_all, proba_list = [], [], []
    for batch in test_loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index)[: batch.batch_size]
        y_all.append(batch.y[: batch.batch_size].cpu().numpy())
        p_all.append(out.argmax(1).cpu().numpy())
        proba_list.append(torch.softmax(out, dim=1).cpu().numpy())
    y_te = np.concatenate(y_all)
    p_te = np.concatenate(p_all)
    proba = np.concatenate(proba_list)
    acc = float((p_te == y_te).mean())
    f1v = f1_score(y_te, p_te, average="macro", zero_division=0)
    c = proba.shape[1]
    try:
        if c == 2:
            aucv = roc_auc_score(y_te, proba[:, 1])
        else:
            aucv = roc_auc_score(
                y_te, proba, multi_class="ovr", average="macro", labels=np.arange(c)
            )
    except ValueError:
        aucv = float("nan")
    return acc, f1v, aucv

acc, f1v, aucv = test()
print(f"Test Accuracy: {acc:.4f} | F1 (macro): {f1v:.4f} | AUC (macro OVR): {aucv:.4f}")


In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
from IPython.display import FileLink, display
from torch_geometric.loader import NeighborLoader

model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best.pth", map_location=device))
torch.save(model.state_dict(), f"{OUTPUT_DIR}/model_weights_twitter.pth")
print("Saved model_weights_twitter.pth")

GNN_NUM_NEIGHBORS = (20, 15, 10)  # match training
E_BATCH = int(os.environ.get("GNN_BATCH_SIZE", "1024")) * 2
n = data.x.size(0)
emb_dim = 128  # == hidden_channels
embeddings = np.zeros((n, emb_dim), dtype=np.float32)
inf_loader = NeighborLoader(
    data,
    num_neighbors=GNN_NUM_NEIGHBORS,
    batch_size=max(E_BATCH, 1024),
    input_nodes=torch.arange(n),
    shuffle=False,
)
model.eval()
with torch.inference_mode():
    for batch in inf_loader:
        b = batch.to(device)
        h = F.relu(model.bn1(model.conv1(b.x, b.edge_index)))
        h = F.relu(model.bn2(model.conv2(h, b.edge_index)))
        h = F.relu(model.bn3(model.conv3(h, b.edge_index)))
        h = F.relu(model.fusion(h))[: b.batch_size].cpu().numpy()
        glob = b.n_id[: b.batch_size].cpu().numpy()
        embeddings[glob] = h

np.save(f"{OUTPUT_DIR}/embeddings_twitter.npy", embeddings)
print("Saved embeddings_twitter.npy")
print("Training successfully completed!")

display(FileLink(f"{OUTPUT_DIR}/model_weights_twitter.pth"))
display(FileLink(f"{OUTPUT_DIR}/embeddings_twitter.npy"))


In [ ]:
import os
import numpy as np
import torch
from sklearn.metrics import f1_score, roc_auc_score
from torch_geometric.loader import NeighborLoader

GNN_NUM_NEIGHBORS = (20, 15, 10)
BATCH = int(os.environ.get("GNN_BATCH_SIZE", "1024")) * 2
BATCH = max(BATCH, 1024)


def _batched_probs_and_pred(idx):
    loader = NeighborLoader(
        data, num_neighbors=GNN_NUM_NEIGHBORS, batch_size=BATCH, input_nodes=idx, shuffle=False
    )
    y_list, p_list, pr_list = [], [], []
    for batch in loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index)[: batch.batch_size]
        y_list.append(batch.y[: batch.batch_size].cpu().numpy())
        p_list.append(out.argmax(1).cpu().numpy())
        pr_list.append(torch.softmax(out, dim=1).cpu().numpy())
    y_ = np.concatenate(y_list)
    p_ = np.concatenate(p_list)
    proba = np.concatenate(pr_list)
    return y_, p_, proba


def f1_auc_from(y_, p_, proba):
    f1_ = f1_score(y_, p_, average="macro", zero_division=0)
    c_ = proba.shape[1]
    try:
        if c_ == 2:
            auc_ = roc_auc_score(y_, proba[:, 1])
        else:
            auc_ = roc_auc_score(
                y_, proba, multi_class="ovr", average="macro", labels=np.arange(c_)
            )
    except ValueError:
        auc_ = float("nan")
    return f1_, auc_

model.eval()
train_idx = data.train_mask.nonzero(as_tuple=True)[0]
test_idx = data.test_mask.nonzero(as_tuple=True)[0]
with torch.inference_mode():
    y_tr, p_tr, pr_tr = _batched_probs_and_pred(train_idx)
    y_te, p_te, pr_te = _batched_probs_and_pred(test_idx)
acc_tr = float((p_tr == y_tr).mean())
acc_te = float((p_te == y_te).mean())
f1_tr, auc_tr = f1_auc_from(y_tr, p_tr, pr_tr)
f1_te, auc_te = f1_auc_from(y_te, p_te, pr_te)
print(f"Train Acc: {acc_tr:.4f} | F1: {f1_tr:.4f} | AUC: {auc_tr:.4f}")
print(f"Test Acc: {acc_te:.4f} | F1: {f1_te:.4f} | AUC: {auc_te:.4f}")
